In [ ]:
!pip install unsloth

# Upgrade Unsloth from the latest repository
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


Found existing installation: unsloth 2025.3.17
Uninstalling unsloth-2025.3.17:
  Successfully uninstalled unsloth-2025.3.17
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-s73p302v/unsloth_a56199b53a504ba7b605eeea8c86e56c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-s73p302v/unsloth_a56199b53a504ba7b605eeea8c86e56c
  Resolved https://github.com/unslothai/unsloth.git to commit 65b8975c5fb65e6c08726f228877ba6b6601f2ba
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.3.17-py3-none-any.whl size=195828 sha256=b9192acc230b69b9b813a5b70b58e59f94bd721ba197448304a182bd77054ab6
  Stored in directory: /tmp/pip-ephem-wheel-cache-1maxc7u5/wheels/d1/17/05/850ab10c33284a4763b0595cd8ea9d01fce6e221cac24b3c01
Successfully built unsloth


In [ ]:
#!pip uninstall bitsandbytes -y
!pip install --upgrade bitsandbytes


In [ ]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")

login(token=hf_token, add_to_git_credential=True)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Meta-Llama-3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16,
    use_auth_token=True
)

print(" Model załadowany i gotowy do użycia!")


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

 Model załadowany i gotowy do użycia!


In [ ]:
import json
from collections import defaultdict
from datasets import Dataset
from transformers import AutoTokenizer

json_path = "sample_data/coursedata 1.json"

with open(json_path, 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

#  Deduplicate courses & merge similar ones
course_dict = defaultdict(lambda: {"course_goals": set(), "course_results": set(), "course_program": set()})

for course in raw_data:
    course_name = course["course_name"]
    course_dict[course_name]["course_goals"].update(course["course_goals"].split(". "))
    course_dict[course_name]["course_results"].update(course["course_results"].split(". "))
    course_dict[course_name]["course_program"].update(course["course_program"].split(". "))

#  Convert sets back to formatted strings
for course_name, details in course_dict.items():
    details["course_goals"] = "\n- " + "\n- ".join(sorted(details["course_goals"]))
    details["course_results"] = "\n- " + "\n- ".join(sorted(details["course_results"]))
    details["course_program"] = "\n- " + "\n- ".join(sorted(details["course_program"]))

#  Load tokenizer
model_id = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
EOS_TOKEN = tokenizer.eos_token  # End of sequence token


def format_training_data():
    structured_data = []
    for course_name, details in course_dict.items():
        prompt = f"""### Instrukcja:
Jesteś asystentem AI przeszkolonym do rekomendowania kursów edukacyjnych na podstawie poziomu studiów użytkownika oraz jego obszaru zainteresowań. Rekomenduj najlepszy kurs dla użytkownika, zapewniając, że:
- Kurs odpowiada poziomowi studiów użytkownika (Studia licencjackie/inżynierskie, Studia magisterskie, Jednolite studia magisterskie, Studia doktoranckie, Studia podyplomowe)
- Oferuje ustrukturyzowaną ścieżkę nauki
- Wyjaśnia kluczowe zagadnienia i oczekiwane rezultaty
- Nie powtarza danych wejściowych użytkownika


### Wejście:
Poziom studiów: {{study_level}}
Obszar studiów: {course_name}

### Odpowiedź:
Najlepszy kurs dla studenta na poziomie {{study_level}}, który studiuje w obszarze {course_name}, to:
"{course_name}"


### Cele kursu:
{details['course_goals']}

#### Oczekiwane rezultaty:
{details['course_results']}

#### Zakres tematów:
{details['course_program']}

"""
        structured_data.append({"text": prompt + EOS_TOKEN})

    return structured_data

#  Convert dataset to Hugging Face format
dataset = Dataset.from_list(format_training_data())


In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset


# Set parameters
max_seq_length = 2048
model_name = "meta-llama/Meta-Llama-3-8B"

# Load model with Unsloth (4-bit quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Apply PEFT (LoRA) for memory-efficient tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
    loftq_config={"quant_type": "fp4"},
)

# Print trainable parameters
print(model.print_trainable_parameters())

# Training arguments
training_args = TrainingArguments(
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_torch_fused",
    weight_decay=0.01,
    warmup_steps=20,
    output_dir="llama-3.2-course-recommender-verB",
    push_to_hub=True,
    hub_model_id="Psylo1226/llama-3.2-course-recommender-verB",
    seed=0,
    report_to="none",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=True,
    args=training_args,
)

==((====))==  Unsloth 2025.3.17: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/config.py:455: UserWarning: `loftq_config` specified but will be ignored when `init_lora_weights` is not 'loftq'.
  warnings.warn("`loftq_config` specified but will be ignored when `init_lora_weights` is not 'loftq'.")


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


Unsloth: Tokenizing ["text"]:   0%|          | 0/9032 [00:00<?, ? examples/s]

Unsloth: Hugging Face's packing is currently buggy - we're disabling it for now!


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,032 | Num Epochs = 3 | Total steps = 1,692
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss
1,1.452000
2,1.494800
3,1.366300
4,1.315800
5,1.236800
6,1.218800
7,1.246000
8,1.073000
9,1.063600
10,1.114900


TrainOutput(global_step=1692, training_loss=0.5785048161644322, metrics={'train_runtime': 16431.3339, 'train_samples_per_second': 1.649, 'train_steps_per_second': 0.103, 'total_flos': 2.485615097644843e+18, 'train_loss': 0.5785048161644322})

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import pipeline, AutoTokenizer

max_seq_length = 2048
model_name = "Psylo1226/llama-3.2-course-recommender-verB"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.for_inference(model).to("cuda")


pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,  # Use float16 for low VRAM usage
    do_sample=True,  # Umożliwia bardziej różnorodne odpowiedzi
    temperature=0.7,  # Balans między kreatywnością a sensownością
    top_p=0.9,  # Ograniczenie generowania do najbardziej prawdopodobnych słów
)

print(" Model loaded successfully on GPU for inference!")

NotImplementedError: Unsloth: No NVIDIA GPU found? Unsloth currently only supports GPUs!

In [ ]:
def recommend_course(topic):
    prompt = f"""### Instrukcja:
Zarekomenduj kurs o tematyce {topic}.

### Odpowiedź:
"""
    output = pipe(prompt, temperature=0.7, top_k=50)
    return output[0]['generated_text']

#Test the model
print(recommend_course("malarstwo"))